In [0]:
dbutils.widgets.text('table', 'src_ath3x')
dbutils.widgets.text('business_date', '2022-01-01')

table = dbutils.widgets.get('table')
business_date = dbutils.widgets.get('business_date')


In [0]:
import sys
import os
import logging
from datetime import date

# -- Make framework modules importable ---------------------------------------
# In Databricks, the repo is mounted at /Workspace/Repos/<user>/etl_framework
REPO_ROOT = "/Workspace/Users/ashok.k.gupta121@gmail.com/etl_framework"
MODULES_PATH = f"{REPO_ROOT}/framework/modules"
SQL_PATH = f"{REPO_ROOT}/framework/modules/sql/visionplus"


# DBTITLE 1,Importing Required Libraries
import pandas as pd
import numpy as np


if MODULES_PATH not in sys.path:
    sys.path.insert(0, MODULES_PATH)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
if SQL_PATH not in sys.path:
    print(SQL_PATH)
    sys.path.insert(0, SQL_PATH)

# -- Import framework modules ------------------------------------------------
from session_manager     import SessionManager
from secret_manager      import SecretManager
from batch_orchestrator  import BatchOrchestrator
from app_config          import EnvironmentConfig, TriggerType
from audit_manager import AuditManager
from connection_manager  import ConnectionManager



In [0]:
# Assuming you have a SparkSession `spark` and a SecretManager `secret_mgr` already initialized,
# and a connection_row object from etl_connection_config for your SQL connection.
env_config = EnvironmentConfig()
session_mgr = SessionManager(env_config)
spark       = session_mgr.get_session()
secret_mgr = SecretManager(spark,'dev-secret-etl')
auditmanager = AuditManager(spark)
connection_row = auditmanager.get_connection_config("CONN_MSSQL_VISIONPLUS")
cm = ConnectionManager(spark, secret_mgr)
import time
retry_attempts = 3
for attempt in range(retry_attempts):
    try:
        df = cm.read_source(connection_row, query_or_table=table)
        df.createOrReplaceTempView("vw_tmp")
        spark.sql(f"delete from {table} where business_date = '{business_date}' ")
        df = spark.sql(f"insert into {table} select *,'{business_date}' from vw_tmp")
        break
    except Exception as e:
        if attempt < retry_attempts - 1:
            time.sleep(5)
        else:
            raise

display(df)
